In [1]:
#examples from GPT:
# {"messages": [{"role": "system", "content": "Marv is a factual chatbot that is also sarcastic."}, {"role": "user", "content": "What's the capital of France?"}, {"role": "assistant", "content": "Paris, as if everyone doesn't know that already."}]}
# {"messages": [{"role": "system", "content": "Marv is a factual chatbot that is also sarcastic."}, {"role": "user", "content": "Who wrote 'Romeo and Juliet'?"}, {"role": "assistant", "content": "Oh, just some guy named William Shakespeare. Ever heard of him?"}]}
# {"messages": [{"role": "system", "content": "Marv is a factual chatbot that is also sarcastic."}, {"role": "user", "content": "How far is the Moon from Earth?"}, {"role": "assistant", "content": "Around 384,400 kilometers. Give or take a few, like that really matters."}]}

In [5]:
import pandas as pd
import json
# import tiktoken # for token counting
import numpy as np
from collections import defaultdict

In [6]:
train_df = pd.read_csv("/home/gridsan/tmackey/cdvae/data/mp_20/train.csv")

In [7]:
train_df.head

<bound method NDFrame.head of        Unnamed: 0.1  Unnamed: 0 material_id  formation_energy_per_atom  \
0                 0       37228  mp-1221227                  -1.637460   
1                 1       19480   mp-974729                  -0.314759   
2                 2       29624  mp-1185360                  -0.193761   
3                 3       38633  mp-1188861                  -0.584694   
4                 4       10889   mp-677272                  -2.474759   
...             ...         ...         ...                        ...   
27131         27131       37856   mp-568116                  -0.988502   
27132         27132       11955   mp-865529                  -0.640955   
27133         27133       26119  mp-1189241                  -0.756019   
27134         27134       30556  mp-1104538                  -0.104870   
27135         27135       32933   mp-756354                  -3.712252   

       band_gap pretty_formula  e_above_hull                       elements  \
0 

In [8]:
import pandas as pd
import json

# Convert DataFrame to desired JSON format
output = []
for _, row in train_df.iterrows():
    entry = {
        "messages": [
            {"role": "user", "content": "these are the x ray diffraction locations " + row["xrd_peak_locations"] + " and these are the x ray diffraction intensities " +row["xrd_peak_intensities"] + " what is the cif file?"},
            {"role": "assistant", "content": str(row["cif"])},
        ]
    }
    output.append(entry)

# Save to JSON
with open('/home/gridsan/tmackey/cdvae/scripts/xrd_train.json', 'w') as f:
    for entry in output:
        f.write(json.dumps(entry) + '\n')

In [12]:
df_copy = train_df.copy()

# Round the xrd_peak_locations and xrd_peak_intensities columns to 2 decimal places
df_copy['xrd_peak_locations'] = df_copy['xrd_peak_locations'].apply(lambda x: [round(val, 2) for val in eval(x)])
df_copy['xrd_peak_intensities'] = df_copy['xrd_peak_intensities'].apply(lambda x: [round(val, 2) for val in eval(x)])

# Display the modified dataframe (optional)
print(df_copy[['xrd_peak_locations', 'xrd_peak_intensities']])

                                      xrd_peak_locations  \
0      [11.87, 16.52, 17.15, 23.2, 23.87, 24.55, 30.1...   
1      [14.07, 19.78, 19.95, 28.23, 28.36, 31.79, 34....   
2      [26.21, 30.35, 43.46, 51.46, 53.93, 63.15, 69....   
3      [14.35, 18.08, 22.0, 27.26, 27.35, 28.53, 28.9...   
4      [14.32, 20.23, 24.87, 24.97, 28.76, 28.87, 32....   
...                                                  ...   
27131  [17.46, 17.73, 20.26, 28.64, 28.98, 33.73, 33....   
27132  [25.18, 29.15, 41.7, 49.33, 51.68, 60.44, 66.5...   
27133  [8.62, 17.3, 22.39, 23.64, 25.58, 26.07, 28.09...   
27134  [9.92, 18.93, 19.92, 21.42, 27.61, 30.07, 33.1...   
27135  [16.59, 16.82, 18.77, 23.71, 29.08, 30.28, 30....   

                                    xrd_peak_intensities  
0      [0.46, 94.05, 0.73, 0.02, 0.44, 0.13, 0.06, 0....  
1      [29.45, 29.52, 12.12, 45.0, 48.77, 19.11, 1.23...  
2      [3.63, 68.25, 100.0, 1.5, 19.06, 16.47, 0.55, ...  
3      [9.99, 12.14, 1.8, 100.0, 39.74, 0.3

In [38]:
# Filter the peaks based on intensities greater than 5
def filter_peaks(locations, intensities):
    filtered_locations = []
    filtered_intensities = []

    for loc, intensity in zip(locations, intensities):
        if intensity > 5:
            filtered_locations.append(loc)
            filtered_intensities.append(intensity)

    return filtered_locations, filtered_intensities

# Apply the filter function to each row of the dataframe
df_copy['filtered_peak_locations'], df_copy['filtered_peak_intensities'] = zip(*df_copy.apply(lambda row: filter_peaks(row['xrd_peak_locations'], row['xrd_peak_intensities']), axis=1))

# Display the modified dataframe (optional)
print(df_copy[['filtered_peak_locations', 'filtered_peak_intensities']])


                                 filtered_peak_locations  \
0      [16.52, 33.4, 34.68, 34.83, 36.06, 36.14, 41.0...   
1      [14.07, 19.78, 19.95, 28.23, 28.36, 31.79, 34....   
2             [30.35, 43.46, 53.93, 63.15, 71.66, 79.77]   
3      [14.35, 18.08, 27.26, 27.35, 29.45, 32.07, 33....   
4      [24.87, 24.97, 28.76, 28.87, 32.24, 32.27, 32....   
...                                                  ...   
27131  [17.46, 17.73, 20.26, 28.64, 28.98, 33.88, 34....   
27132  [25.18, 29.15, 41.7, 49.33, 51.68, 60.44, 66.5...   
27133  [8.62, 17.3, 23.64, 28.09, 32.45, 32.52, 34.35...   
27134  [19.92, 30.07, 33.1, 35.77, 38.41, 38.93, 40.4...   
27135  [18.77, 23.71, 30.28, 30.29, 33.55, 33.66, 34....   

                               filtered_peak_intensities  
0      [94.05, 23.8, 13.93, 6.3, 31.78, 15.92, 100.0,...  
1      [29.45, 29.52, 12.12, 45.0, 48.77, 19.11, 35.1...  
2              [68.25, 100.0, 19.06, 16.47, 25.1, 33.91]  
3      [9.99, 12.14, 100.0, 39.74, 73.47, 7

In [41]:
# Convert DataFrame to desired JSON format
filtered_output = []
for _, row in df_copy.iterrows():
    entry = {
        "messages": [
            {"role": "user", "content": "these are the x ray diffraction locations " + str(row["filtered_peak_locations"]) + " and these are the x ray diffraction intensities " + str(row["filtered_peak_intensities"]) + " what is the cif file?"},
            {"role": "assistant", "content": str(row["cif"])},
        ]
    }
    filtered_output.append(entry)

# Save to JSON
with open('/home/gridsan/tmackey/cdvae/scripts/xrd_train_filtered.json', 'w') as f:
    for entry in filtered_output:
        f.write(json.dumps(entry) + '\n')

data_path = "/home/gridsan/tmackey/cdvae/scripts/xrd_train_filtered.json"
dataset = []
# Load the dataset
with open(data_path, 'r', encoding='utf-8') as f:
    dataset = [json.loads(line) for line in f]

# Initial dataset stats
print("Num examples:", len(dataset))
print("First example:")
for message in dataset[0]["messages"]:
    print(message)

In [43]:
# Convert DataFrame to desired JSON format
filtered_space_group_output = []
for _, row in df_copy.iterrows():
    entry = {
        "messages": [
            {"role": "user", "content": "x-ray diffraction peak locations " + str(row["filtered_peak_locations"]) + " x-ray diffraction peak intensities: " + str(row["filtered_peak_intensities"]) + " what is the space group?"},
            {"role": "assistant", "content": str(row["spacegroup.number"])},
        ]
    }
    filtered_space_group_output.append(entry)

# Save to JSON
with open('/home/gridsan/tmackey/cdvae/scripts/xrd_train_filtered_space_group.json', 'w') as f:
    for entry in filtered_space_group_output:
        f.write(json.dumps(entry) + '\n')

data_path = "/home/gridsan/tmackey/cdvae/scripts/xrd_train_filtered_space_group.json"
dataset = []
# Load the dataset
with open(data_path, 'r', encoding='utf-8') as f:
    dataset = [json.loads(line) for line in f]

# Initial dataset stats
print("Num examples:", len(dataset))
print("First example:")
for message in dataset[0]["messages"]:
    print(message)

Num examples: 27136
First example:
{'role': 'user', 'content': 'x-ray diffraction peak locations [16.52, 33.4, 34.68, 34.83, 36.06, 36.14, 41.08, 41.26, 52.94, 53.14, 57.78, 57.81, 61.19, 61.44, 68.35, 70.16, 71.53, 71.69, 71.83, 76.49, 76.69, 89.14] x-ray diffraction peak intensities: [94.05, 23.8, 13.93, 6.3, 31.78, 15.92, 100.0, 47.58, 10.85, 5.14, 13.83, 27.23, 13.62, 25.29, 7.7, 5.75, 5.24, 5.2, 5.04, 11.64, 5.5, 6.85] what is the space group?'}
{'role': 'assistant', 'content': '8'}


In [31]:
# Round the xrd_peak_locations and xrd_peak_intensities columns to 2 decimal places
df_copy['disc_sim_xrd'] = df_copy['disc_sim_xrd'].apply(lambda x: [round(float(val), 1) for val in x.strip('[]').split()])

# Display the modified dataframe (optional)
print(df_copy['disc_sim_xrd'])

0        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
1        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
2        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
3        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
4        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
                               ...                        
27131    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
27132    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
27133    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
27134    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
27135    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
Name: disc_sim_xrd, Length: 27136, dtype: object


In [34]:
# using the simulated diffraction pattern 
dis_sim_results = []
for _, row in df_copy.iterrows():
    entry = {
        "messages": [
            {"role": "user", "content": "these is an x-ray diffraction pattern from 5 to 75 in steps of 0.35 " + str(row['disc_sim_xrd']) + " what is the cif file?"},
            {"role": "assistant", "content": str(row["cif"])},
        ]
    }
    dis_sim_results.append(entry)

# Save to JSON
with open('/home/gridsan/tmackey/cdvae/scripts/xrd_train_disc_sim.json', 'w') as f:
    for entry in dis_sim_results:
        f.write(json.dumps(entry) + '\n')

In [35]:
data_path = "/home/gridsan/tmackey/cdvae/scripts/xrd_train_disc_sim.json"
dataset = []
# Load the dataset
with open(data_path, 'r', encoding='utf-8') as f:
    dataset = [json.loads(line) for line in f]

# Initial dataset stats
print("Num examples:", len(dataset))
print("First example:")
for message in dataset[0]["messages"]:
    print(message)

Num examples: 27136
First example:
{'role': 'user', 'content': 'these is an x-ray diffraction pattern from 5 to 75 in steps of 0.35 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 94.1, 0.0, 0.7, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.4, 0.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.4, 0.0, 0.0, 0.0, 23.8, 0.0, 0.0, 0.0, 13.9, 0.0, 0.2, 0.0, 31.8, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 100.0, 47.6, 0.0, 1.1, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 2.5, 0.0, 0.0, 0.0, 0.0, 0.4, 0.0, 0.1, 0.1, 0.0, 0.0, 0.8, 0.6, 0.0, 0.1, 0.0, 0.0, 0.1, 0.0, 0.3, 0.0, 0.0, 0.0, 0.0, 10.9, 5.1, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.3, 0.0, 27.2, 0.0, 0.0, 0.1, 0.0, 0.1, 0.2, 0.0, 0.0, 0.3, 25.3, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.9, 3.6, 0.2, 0.

In [10]:
# Format error checks
format_errors = defaultdict(int)

for ex in dataset:
    if not isinstance(ex, dict):
        format_errors["data_type"] += 1
        continue
        
    messages = ex.get("messages", None)
    if not messages:
        format_errors["missing_messages_list"] += 1
        continue
        
    for message in messages:
        if "role" not in message or "content" not in message:
            format_errors["message_missing_key"] += 1
        
        if any(k not in ("role", "content", "name", "function_call") for k in message):
            format_errors["message_unrecognized_key"] += 1
        
        if message.get("role", None) not in ("system", "user", "assistant", "function"):
            format_errors["unrecognized_role"] += 1
            
        content = message.get("content", None)
        function_call = message.get("function_call", None)
        
        if (not content and not function_call) or not isinstance(content, str):
            format_errors["missing_content"] += 1
    
    if not any(message.get("role", None) == "assistant" for message in messages):
        format_errors["example_missing_assistant_message"] += 1

if format_errors:
    print("Found errors:")
    for k, v in format_errors.items():
        print(f"{k}: {v}")
else:
    print("No errors found")

No errors found


In [11]:
# Warnings and tokens counts
n_missing_system = 0
n_missing_user = 0
n_messages = []
convo_lens = []
assistant_message_lens = []

for ex in dataset:
    messages = ex["messages"]
    if not any(message["role"] == "system" for message in messages):
        n_missing_system += 1
    if not any(message["role"] == "user" for message in messages):
        n_missing_user += 1
    n_messages.append(len(messages))
    convo_lens.append(num_tokens_from_messages(messages))
    assistant_message_lens.append(num_assistant_tokens_from_messages(messages))
    
print("Num examples missing system message:", n_missing_system)
print("Num examples missing user message:", n_missing_user)
print_distribution(n_messages, "num_messages_per_example")
print_distribution(convo_lens, "num_total_tokens_per_example")
print_distribution(assistant_message_lens, "num_assistant_tokens_per_example")
n_too_long = sum(l > 4096 for l in convo_lens)
print(f"\n{n_too_long} examples may be over the 4096 token limit, they will be truncated during fine-tuning")

NameError: name 'num_tokens_from_messages' is not defined